In [ ]:
import random
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
import os

# ==========================================
# KONFIGURASI FILE & HEADER
# ==========================================
IN_FILE = "dataset_minoritas_anti_bot_final.csv"
OUT_FILE = "rescraped_candidates.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

# ==========================================
# FUNGSI INTI
# ==========================================
def unescape(s) -> str:
    s = str(s)
    s = re.sub(r"\\u([0-9a-fA-F]{4})", lambda m: chr(int(m.group(1), 16)), s)
    s = s.replace("\\n", " ").replace("\\t", " ").replace('\\"', '"').replace("\\/", "/")
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip(' "')

def is_doctor(el) -> bool:
    title = (el.get("doctor-title-small") or "").strip().lower()
    nic = (el.get("by-nicename") or "").lower()
    byd = (el.get("by-doctor") or "").strip().lower()
    
    return (title.startswith("dokter")
            or nic.startswith(("dr-", "drg-"))
            or byd.startswith(("dr.", "drg.", "dr ", "drg ")))

def scrape(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    if r.status_code != 200:
        return "", "", f"http_{r.status_code}"
    
    soup = BeautifulSoup(r.text, "html.parser")
    dt = soup.find("detail-topic")
    
    keluhan = unescape(dt.get("member-topic-content")) if dt else ""
    if dt and len(keluhan) < 10:
        keluhan = unescape(dt.get("member-topic-title")) or keluhan
        
    docs = [el for el in soup.find_all("doctor-topic") if is_doctor(el)]
    if not docs:
        return keluhan, "", "no_doctor"
        
    jawaban = " ".join(unescape(el.get("doctor-topic-content")) for el in docs).strip()
    return keluhan, jawaban, "ok"

# ==========================================
# EKSEKUSI NOTEBOOK
# ==========================================
if not os.path.exists(IN_FILE):
    print(f"FATAL: File '{IN_FILE}' tidak ditemukan di direktori aktif.")
else:
    df = pd.read_csv(IN_FILE)
    rows = []
    
    print(f"MEMULAI RE-SCRAPING MENDALAM UNTUK {len(df)} BARIS DATA")
    print("=" * 70)
    
    for n, (_, row) in enumerate(df.iterrows(), 1):
        url = row["URL"]
        keluhan, jawaban, status = "", "", "err:unknown"
        
        for attempt in range(2):
            try:
                keluhan, jawaban, status = scrape(url)
                break
            except Exception as e:
                status = f"err:{type(e).__name__}"
                time.sleep(10)
                
        # Menghapus referensi row_id yang menyebabkan fatal error
        # Menggunakan .get() sebagai perlindungan ekstra terhadap nama kolom
        rows.append({
            "Poli_Tujuan": row.get("Poli_Tujuan", "Tidak Diketahui"), 
            "URL": url,
            "Keluhan": keluhan, 
            "Jawaban_Dokter": jawaban,
            "Jawaban_Lama": row.get("Jawaban_Dokter", ""), 
            "status": status,
        })
        
        print(f"[{n}/{len(df)}] {status:9s} | {url.split('/topic/')[-1][:45]}")
        time.sleep(random.uniform(2, 4))

    # Penyimpanan hasil akhir
    out = pd.DataFrame(rows)
    out.to_csv(OUT_FILE, index=False, encoding="utf-8")
    
    print("\n" + "=" * 70)
    print("RINGKASAN STATUS RE-SCRAPING")
    print("=" * 70)
    print(out["status"].value_counts().to_string())
    print(f"\n[SUKSES] Dataset tervalidasi tersimpan di: '{OUT_FILE}'")

MEMULAI RE-SCRAPING MENDALAM UNTUK 5398 BARIS DATA
[1/5398] ok        | jenis-olahraga-apa-yang-aman-dan-yang-dilaran
[2/5398] ok        | apakah-nyeri-dan-benjolan-di-anus-tanda-ambei
[3/5398] ok        | wasir-berdarah-saat-hamil-7-bulan--apakah-ber
[4/5398] ok        | apa-risiko-jika-ambeien-dibiarkan-membesar-
[5/5398] ok        | apakah-olahraga-angkat-beban-bisa-sebabkan-am
[6/5398] ok        | makanan-apa-yang-harus-dihindari-penderita-am
[7/5398] ok        | obat-wasir-apa-yang-aman-untuk-ibu-hamil-
[8/5398] ok        | apakah-olahraga-bisa-membantu-mengatasi-ambei
[9/5398] ok        | bagaimana-mengobati-ambeien-ringan-di-rumah-
[10/5398] ok        | bagaimana-cara-mengatasi-wasir-saat-hamil-
[11/5398] ok        | bagaimana-mencegah-dan-mengatasi-wasir-pada-p
[12/5398] ok        | bagaimana-cara-penanganan-yang-aman-untuk-lan
[13/5398] ok        | bagaimana-membedakan-ambeien-dalam-dan-luar-d
[14/5398] ok        | bagaimana-cara-mengatasi-ambein-tanpa-operasi
[15/5398] ok    

: 